# Additional Experiment — Supplement Configs (Activation Extraction)

Mirrors `experiment.ipynb`, but for **vote buckets that already have generations + judge
votes** and were unused by the main 3-class experiment. These are re-labeled directly from
the existing `*_full.csv` — **no knowledge test, no generation, no judging**. Scope of this
notebook: build the supplement probe dataset and **extract activations**. PCA + probing are
deferred to the final (commented) section.

New classes (defined by `SUPPLEMENT_LABEL_RULES` in `utils/settings.py`):

| label | bucket (config / votes_correct) | meaning |
|---|---|---|
| `natural_deception` | A / 0 | passed knowledge test, neutral prompt, all-wrong |
| `capable_failed` | B / 6 | failed knowledge test, neutral prompt, all-right |
| `deception_rejection` | C / 6 | passed knowledge test, deceptive prompt, all-right |

**To change any experiment parameter, edit `utils/settings.py` — never hard-code it here:**
- model → `MODEL_ID` (qwen2.5 / qwen3 / gemma-4); prompt/thinking variant → `RUN_SLUG`
- which buckets map to which labels → `SUPPLEMENT_LABEL_RULES`
- PCA size / probe hyperparams → `PCA_K`, `N_SPLITS`, `MAX_ITER`, `RANDOM_STATE`

Switching to qwen3 / gemma-4 later needs no change here: `split_thinking_responses`
already strips their thinking blocks, and all paths derive from settings.

## Part 1: Setup & Load Model

In [ ]:
import os
os.environ["VLLM_ENABLE_V1_MULTIPROCESSING"] = "0"

import random
import numpy as np
import pandas as pd
import torch
import matplotlib
matplotlib.use("Agg")  # save plots to files only — do not display inline
import matplotlib.pyplot as plt
from pathlib import Path
from transformers import AutoTokenizer, AutoModelForCausalLM
import warnings
warnings.filterwarnings("ignore")

# Settings — single source of truth for all paths, constants, and hyperparameters.
# Change MODEL_ID / RUN_SLUG / SUPPLEMENT_LABEL_RULES there, never in this notebook.
from utils.settings import *

# Utils (reused — this notebook contains no bespoke extraction/probing loops)
from utils.analysis import build_probe_dataset, split_thinking_responses, run_pca_reduction
from utils.activation import run_extract_activations, LABEL_MAP
from utils.probe import probe_all_layers
from utils.plotting import plot_macro_f1

# Reproducibility
random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)
torch.manual_seed(RANDOM_STATE)

# Output directories for the supplement run (distinct filenames live in these folders)
for d in [SUPPLEMENT_PROBE_DATASET_PATH.parent, SUPPLEMENT_ACTIVATIONS_PATH.parent]:
    d.mkdir(parents=True, exist_ok=True)

print(f"Model:    {MODEL_ID}")
print(f"Run slug: {RUN_SLUG or '(none)'}")
print(f"Supplement rules: {SUPPLEMENT_LABEL_RULES}")
print("Reading judged data from:")
print(f"  {TRUTHFULQA_FULL_PATH}")
print(f"  {MMLU_FULL_PATH}")
print(f"\nDevice: {DEVICE}")
if DEVICE == "cuda" and torch.cuda.is_available():
    print(f"GPU:  {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, token=HF_READ_TOKEN)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    dtype=torch.bfloat16,
    device_map="auto",
    max_memory={0: "22GiB", "cpu": "120GiB"},
    offload_folder="outputs/offload",
    token=HF_READ_TOKEN,
)
model.eval()

# Support different config schemas (e.g., Gemma family and others).
cfg = getattr(model.config, "text_config", model.config)
N_LAYERS   = getattr(cfg, "num_hidden_layers", getattr(cfg, "num_layers", None))
HIDDEN_DIM = getattr(cfg, "hidden_size", getattr(cfg, "d_model", None))

print(f"Loaded: {MODEL_ID}")
print(f"Layers: {N_LAYERS}, hidden_dim: {HIDDEN_DIM}")

## Part 2: Load Existing Judged Data

No knowledge test, generation, or judging is run — the supplement configs are re-labeled
from the existing `truthfulQA_full.csv` / `mmlu_full.csv`, which already hold every
`(config, votes_correct)` bucket.

In [ ]:
tqa_full  = pd.read_csv(TRUTHFULQA_FULL_PATH)
mmlu_full = pd.read_csv(MMLU_FULL_PATH)
print(f"tqa_full : {tqa_full.shape}")
print(f"mmlu_full: {mmlu_full.shape}")

# Sanity check: rows each supplement rule will select (TruthfulQA + MMLU)
for config, vc, label in SUPPLEMENT_LABEL_RULES:
    n = int(
        ((tqa_full["config"] == config)  & (tqa_full["votes_correct"] == vc)).sum()
        + ((mmlu_full["config"] == config) & (mmlu_full["votes_correct"] == vc)).sum()
    )
    print(f"  {label:20s} (config {config} / votes_correct {vc}): {n} rows")

## Part 3: Build Supplement Probe Dataset

Factual rows only — the new classes have no social scenario pairs, so
`include_social=False`. Labels come from `SUPPLEMENT_LABEL_RULES`.

In [ ]:
probe_dataset = build_probe_dataset(
    tqa_full, mmlu_full, None,
    SUPPLEMENT_PROBE_DATASET_PATH,
    rules=SUPPLEMENT_LABEL_RULES,
    include_social=False,
)

# Split thinking/answer: no-op for qwen2.5; strips <think>/gemma blocks for qwen3/gemma.
probe_dataset_split = split_thinking_responses(
    probe_dataset,
    save_path=SUPPLEMENT_PROBE_DATASET_SPLIT_PATH,
)

## Part 4: Extract Activations  *(endpoint of this notebook)*

Extracts the last-token hidden state per layer for the supplement rows only.

- Distinct filenames (`activations_new_config.npy`, `labels_new_config.npy`) sit alongside
  the 3-class files without overwriting them.
- `run_extract_activations` derives its HuggingFace download name from the path, so the
  3-class `activations.npy` is never pulled by mistake; on the first run nothing is found
  on the Hub and it extracts locally. Upload afterwards to reuse next time, or pass
  `hf_repo=""` to skip the Hub entirely.

In [ ]:
_df = probe_dataset_split.copy()
_df["response"] = _df["response_answer"]

activations_arr, labels_arr = run_extract_activations(
    _df, model, tokenizer, DEVICE,
    SUPPLEMENT_ACTIVATIONS_PATH, SUPPLEMENT_LABELS_PATH, SUPPLEMENT_ACTIVATIONS_CHECKPOINT_PATH,
    HF_ACTIVATIONS_REPO, HF_READ_TOKEN, CHECKPOINT_EVERY,
)
print(f"activations: {activations_arr.shape}")
print("Label counts:",
      {k: int((labels_arr == v).sum()) for k, v in LABEL_MAP.items() if (labels_arr == v).any()})

## Part 5 (deferred): PCA + Probing

Scope of this notebook is activation extraction. When the probing design is decided,
uncomment below. The supplement labels are plain strings, so the generic multi-class
probes in `utils/probe.py` work unchanged (no probe.py edits needed). To probe the new
classes *together with* the original three, load the 3-class `activations.npy` / `labels.npy`
and `np.concatenate` before PCA.

In [ ]:
# labels_str = np.array([{v: k for k, v in LABEL_MAP.items()}[i] for i in labels_arr])

# acts_reduced = run_pca_reduction(
#     activations_arr, PCA_K,
#     SUPPLEMENT_ACTIVATIONS_PCA_PATH, SUPPLEMENT_PCA_COMPONENTS_PATH, SUPPLEMENT_PCA_VARIANCE_PATH,
#     hf_repo="", hf_token=HF_READ_TOKEN,   # "" -> always compute locally for this set
# )

# _probe_dir = OUTPUT_DIR / "new_config_3way_lr"
# (_probe_dir / "figures").mkdir(parents=True, exist_ok=True)
# results_lr = probe_all_layers(
#     acts_reduced, labels_str,
#     n_splits=N_SPLITS, max_iter=MAX_ITER, random_state=RANDOM_STATE,
#     output_path=_probe_dir / "probe_results.csv",
#     checkpoint_path=_probe_dir / "checkpoint.pkl",
# )
# plot_macro_f1(results_lr, _probe_dir / "figures" / "macro_f1.png",
#               title="Supplement configs: multi-class LR Macro-F1")